# Bright LEO Photometry
Written by Kiyoaki Okudaira<br>
*Kyushu University Hanada Lab / University of Washington / IAU CPS SatHub<br>
(okudaira.kiyoaki.528@s.kyushu-u.ac.jp or kiyoaki@uw.edu)<br>
<br>
Measure lightcurve of bright LEO object.<br>
<br>
**History**<br>
coding 2026-07-05 : 1st coding<br>
<br>
(c) 2026 Kiyoaki Okudaira - Kyushu University Hanada Lab (SSDL) / University of Washington / IAU CPS SatHub

### Parameters
**Target information**

In [ ]:
object_name   = "HORN-R"    # object name | str or None
intl_des      = "0000-0000" # international designator | str or None
norad_id      = None        # NORAD catalog ID | str or None

observer_name = "Kyushu University Hanada Lab (SSDL)"
observatory   = "KUBO Kyushu University Bull's-eye Telescope"

**Measurement settings**

In [ ]:
# aperture radius for object
aperture_radius = 30  # [pixel] | int or float

# aperture radius for background
annulus_r_in    = 36  # [pixel] | int or float
annulus_r_out   = 42  # [pixel] | int or float

**Detection settings**

In [ ]:
# Re-run full-screen detection if the center-of-gravity calculation fails
redetect_on_failure = True # bool

### Import and initial settings
**PATH settings**

In [ ]:
PATH_base  = "/Users/kiyoaki/VScode/satphotometry_photometry/"
PATH_fits  = "/Volumes/SSD-PEU4A/Images/2026-07-02/SharpCap Captures/HORN-R/21_50_49"

PATH_output = PATH_base + "output/"
PATH_magzero = PATH_output + "magzero/magzero_2026-07-02T125622_GAIN56.json"

**Standard libraries**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import json
from pathlib import Path
import warnings
from datetime import datetime, timezone

from astropy.io import fits
from astropy.stats import sigma_clipped_stats, SigmaClip
from astropy.table import Table
from astropy.time import Time, TimeDelta

from scipy import ndimage

from astropy.stats import SigmaClip
from photutils.aperture import (
    CircularAperture,
    CircularAnnulus,
    ApertureStats,
    aperture_photometry,
)
from photutils.centroids import centroid_2dg

### Object detection

#### Measurement

In [ ]:
# ============================================================
# Get the list of FITS files
# ============================================================

def detect_brightest_point_source(
    image,
    detection_sigma=8.0,
    edge_sigma=5.0,
    cutout_half_size=30,
    min_pixels=5,
):
    """
    画像中で最も明るいほぼ点像の天体を検出し、
    輝度加重重心と点像半径を求める。

    Parameters
    ----------
    image : 2D ndarray
        FITS画像データ。

    detection_sigma : float
        天体候補領域を抽出する際の背景ノイズに対する閾値。

    edge_sigma : float
        Sobelエッジを抽出する際の閾値。
        小さくすると弱いエッジまで拾う。

    cutout_half_size : int
        最も明るい画素の周囲に切り出す半幅 [pixel]。

    min_pixels : int
        天体候補として必要な最小画素数。

    Returns
    -------
    result : dict
        以下を含む辞書。

        x_centroid : float
            画像全体におけるx方向重心 [pixel]。

        y_centroid : float
            画像全体におけるy方向重心 [pixel]。

        radius : float
            エッジ画素までの距離の中央値から求めた半径 [pixel]。

        radius_std : float
            エッジ半径のばらつき [pixel]。

        equivalent_radius : float
            検出領域の面積から求めた等価半径 [pixel]。

        edge_x, edge_y : ndarray
            画像全体におけるエッジ画素座標。

        source_mask : ndarray
            切り出し画像内の人工天体候補マスク。

        cutout : ndarray
            人工天体周辺の切り出し画像。

        bounds : tuple
            切り出し範囲 (x_min, x_max, y_min, y_max)。
    """

    image = np.asarray(image, dtype=float)

    if image.ndim != 2:
        raise ValueError("imageは2次元配列である必要があります。")

    # --------------------------------------------------------
    # NaN、infを背景中央値で置換
    # --------------------------------------------------------

    finite_mask = np.isfinite(image)

    if not np.any(finite_mask):
        raise ValueError("画像中に有限値がありません。")

    global_median = np.nanmedian(image[finite_mask])

    clean_image = image.copy()
    clean_image[~finite_mask] = global_median

    # --------------------------------------------------------
    # 画像全体の背景統計
    # --------------------------------------------------------

    mean, median, std = sigma_clipped_stats(
        clean_image,
        sigma=3.0,
        maxiters=5,
    )

    if not np.isfinite(std) or std <= 0:
        raise ValueError("背景ノイズの標準偏差を推定できませんでした。")

    # --------------------------------------------------------
    # 最も明るい画素を探す
    # --------------------------------------------------------

    brightest_index = np.nanargmax(clean_image)
    brightest_y, brightest_x = np.unravel_index(
        brightest_index,
        clean_image.shape,
    )

    # --------------------------------------------------------
    # 明るい画素の周囲を切り出す
    # --------------------------------------------------------

    ny, nx = clean_image.shape

    x_min = max(0, brightest_x - cutout_half_size)
    x_max = min(nx, brightest_x + cutout_half_size + 1)

    y_min = max(0, brightest_y - cutout_half_size)
    y_max = min(ny, brightest_y + cutout_half_size + 1)

    cutout = clean_image[y_min:y_max, x_min:x_max].copy()

    # --------------------------------------------------------
    # 切り出し領域の背景を再推定
    # --------------------------------------------------------

    cut_mean, cut_median, cut_std = sigma_clipped_stats(
        cutout,
        sigma=3.0,
        maxiters=5,
    )

    if not np.isfinite(cut_std) or cut_std <= 0:
        cut_median = median
        cut_std = std

    background_subtracted = cutout - cut_median

    # --------------------------------------------------------
    # 人工天体候補の領域を抽出
    # --------------------------------------------------------

    source_threshold = detection_sigma * cut_std
    threshold_mask = background_subtracted > source_threshold

    # 小さな穴を埋める
    threshold_mask = ndimage.binary_fill_holes(threshold_mask)

    # 1画素程度の隙間をつなぐ
    threshold_mask = ndimage.binary_closing(
        threshold_mask,
        structure=np.ones((3, 3), dtype=bool),
    )

    # 連結領域に分割
    labels, number_of_labels = ndimage.label(threshold_mask)

    if number_of_labels == 0:
        raise RuntimeError(
            "人工天体候補を検出できませんでした。"
            "detection_sigmaを小さくしてください。"
        )

    # --------------------------------------------------------
    # 最も明るい画素を含む連結領域を選択
    # --------------------------------------------------------

    local_brightest_x = brightest_x - x_min
    local_brightest_y = brightest_y - y_min

    target_label = labels[local_brightest_y, local_brightest_x]

    if target_label == 0:
        # 最も明るい画素が閾値領域に入っていない場合は、
        # 総フラックスが最大の領域を選択
        label_fluxes = []

        for label_number in range(1, number_of_labels + 1):
            region = labels == label_number
            flux = np.sum(
                np.clip(background_subtracted[region], 0, None)
            )
            label_fluxes.append(flux)

        target_label = np.argmax(label_fluxes) + 1

    source_mask = labels == target_label

    if np.count_nonzero(source_mask) < min_pixels:
        raise RuntimeError(
            f"検出領域が小さすぎます："
            f"{np.count_nonzero(source_mask)} pixels"
        )

    # --------------------------------------------------------
    # 輝度加重重心
    # --------------------------------------------------------

    yy, xx = np.indices(cutout.shape)

    weights = np.clip(background_subtracted, 0, None)
    weights[~source_mask] = 0.0

    total_weight = np.sum(weights)

    if total_weight <= 0:
        raise RuntimeError("重心計算に使用できる正のフラックスがありません。")

    x_centroid_local = np.sum(xx * weights) / total_weight
    y_centroid_local = np.sum(yy * weights) / total_weight

    x_centroid = x_centroid_local + x_min
    y_centroid = y_centroid_local + y_min

    # --------------------------------------------------------
    # Sobelフィルタによるエッジ検出
    # --------------------------------------------------------

    # ノイズ抑制のため軽くGaussian平滑化
    smoothed = ndimage.gaussian_filter(
        background_subtracted,
        sigma=1.0,
    )

    gradient_x = ndimage.sobel(smoothed, axis=1, mode="reflect")
    gradient_y = ndimage.sobel(smoothed, axis=0, mode="reflect")

    gradient_magnitude = np.hypot(gradient_x, gradient_y)

    # 背景領域における勾配ノイズを推定
    background_region = ~ndimage.binary_dilation(
        source_mask,
        iterations=2,
    )

    gradient_background = gradient_magnitude[background_region]

    if gradient_background.size > 0:
        _, gradient_median, gradient_std = sigma_clipped_stats(
            gradient_background,
            sigma=3.0,
            maxiters=5,
        )
    else:
        gradient_median = np.median(gradient_magnitude)
        gradient_std = np.std(gradient_magnitude)

    edge_threshold = gradient_median + edge_sigma * gradient_std

    edge_mask_raw = gradient_magnitude > edge_threshold

    # 人工天体候補の近傍だけに限定
    source_neighborhood = ndimage.binary_dilation(
        source_mask,
        iterations=2,
    )

    edge_mask = edge_mask_raw & source_neighborhood

    # source_mask自体の境界も求める
    eroded_mask = ndimage.binary_erosion(source_mask)
    mask_boundary = source_mask & ~eroded_mask

    # Sobelエッジが少なすぎる場合に備えてマスク境界を追加
    edge_mask = edge_mask | mask_boundary

    edge_y_local, edge_x_local = np.nonzero(edge_mask)

    if len(edge_x_local) == 0:
        raise RuntimeError("人工天体のエッジを検出できませんでした。")

    # --------------------------------------------------------
    # 重心からエッジまでの距離を半径として求める
    # --------------------------------------------------------

    edge_distances = np.sqrt(
        (edge_x_local - x_centroid_local) ** 2
        + (edge_y_local - y_centroid_local) ** 2
    )

    # 異常に内側または外側のエッジを除外
    distance_median = np.median(edge_distances)
    distance_mad = np.median(
        np.abs(edge_distances - distance_median)
    )

    if distance_mad > 0:
        robust_sigma = 1.4826 * distance_mad

        valid_edge = (
            np.abs(edge_distances - distance_median)
            < 3.0 * robust_sigma
        )

        filtered_distances = edge_distances[valid_edge]
        edge_x_local = edge_x_local[valid_edge]
        edge_y_local = edge_y_local[valid_edge]
    else:
        filtered_distances = edge_distances

    radius = np.median(filtered_distances)
    radius_std = np.std(filtered_distances)

    # --------------------------------------------------------
    # 面積から求める等価半径
    # --------------------------------------------------------

    source_area = np.count_nonzero(source_mask)
    equivalent_radius = np.sqrt(source_area / np.pi)

    # エッジ座標を画像全体の座標へ変換
    edge_x = edge_x_local + x_min
    edge_y = edge_y_local + y_min

    return {
        "x_centroid": float(x_centroid),
        "y_centroid": float(y_centroid),
        "radius": float(radius),
        "radius_std": float(radius_std),
        "equivalent_radius": float(equivalent_radius),
        "source_area": int(source_area),
        "peak_value": float(clean_image[brightest_y, brightest_x]),
        "background": float(cut_median),
        "background_std": float(cut_std),
        "edge_x": edge_x,
        "edge_y": edge_y,
        "source_mask": source_mask,
        "edge_mask": edge_mask,
        "cutout": cutout,
        "background_subtracted": background_subtracted,
        "bounds": (x_min, x_max, y_min, y_max),
    }

def detect_source_around_previous_position(
    image,
    previous_x,
    previous_y,
    search_half_size=50,
    detection_sigma=8.0,
    edge_sigma=5.0,
    detection_cutout_half_size=30,
):
    """
    直前フレームの人工天体位置周辺を切り出し、
    その小領域内で detect_brightest_point_source() を実行する。

    Parameters
    ----------
    image : ndarray
        元の2次元画像。

    previous_x, previous_y : float
        直前フレームでの人工天体位置。

    search_half_size : int
        直前位置を中心に探索する半幅 [pixel]。
        実際の探索領域サイズは、およそ
        (2 * search_half_size + 1)^2 pixel。

    detection_sigma : float
        detect_brightest_point_source() に渡す検出閾値。

    edge_sigma : float
        detect_brightest_point_source() に渡すエッジ検出閾値。

    detection_cutout_half_size : int
        detect_brightest_point_source() 内部で使う切り出し半幅。

    Returns
    -------
    result : dict
        元画像座標系へ変換した検出結果。
    """

    image = np.asarray(image, dtype=np.float64)

    if image.ndim != 2:
        raise ValueError(
            f"画像が2次元ではありません: shape={image.shape}"
        )

    try:
        previous_x = float(previous_x)
        previous_y = float(previous_y)
    except (TypeError, ValueError) as error:
        raise TypeError(
            "直前位置をfloatへ変換できません。"
            f" previous_x={previous_x!r},"
            f" previous_y={previous_y!r}"
        ) from error

    if not (
        np.isfinite(previous_x)
        and np.isfinite(previous_y)
    ):
        raise ValueError(
            "直前位置が有限値ではありません。"
        )

    ny, nx = image.shape

    x_center = int(round(previous_x))
    y_center = int(round(previous_y))

    x_min = max(0, x_center - search_half_size)
    x_max = min(nx, x_center + search_half_size + 1)

    y_min = max(0, y_center - search_half_size)
    y_max = min(ny, y_center + search_half_size + 1)

    local_image = image[
        y_min:y_max,
        x_min:x_max
    ]

    if local_image.size == 0:
        raise ValueError(
            "局所探索領域が空です。"
        )

    local_result = detect_brightest_point_source(
        local_image,
        detection_sigma=detection_sigma,
        edge_sigma=edge_sigma,
        cutout_half_size=detection_cutout_half_size,
    )

    x_local = float(
        np.asarray(
            local_result["x_centroid"]
        ).squeeze()
    )

    y_local = float(
        np.asarray(
            local_result["y_centroid"]
        ).squeeze()
    )

    if not (
        np.isfinite(x_local)
        and np.isfinite(y_local)
    ):
        raise ValueError(
            "局所探索で得られた位置が有限値ではありません。"
        )

    # 局所画像座標から元画像座標へ変換
    x_global = float(x_min + x_local)
    y_global = float(y_min + y_local)

    if not (
        0 <= x_global < nx
        and 0 <= y_global < ny
    ):
        raise ValueError(
            "変換後の位置が元画像の範囲外です。"
        )

    result = dict(local_result)

    result["x_centroid_local"] = x_local
    result["y_centroid_local"] = y_local

    result["x_centroid"] = x_global
    result["y_centroid"] = y_global

    result["search_x_min"] = x_min
    result["search_x_max"] = x_max
    result["search_y_min"] = y_min
    result["search_y_max"] = y_max

    return result

def get_fits_files(directory):
    """
    Return FITS files directly contained in the specified
    directory, excluding hidden files.

    Files are sorted alphabetically by filename.

    Parameters
    ----------
    directory : str or pathlib.Path
        Directory containing FITS files.

    Returns
    -------
    fits_files : list[pathlib.Path]
        Sorted list of FITS file paths.
    """

    directory = Path(directory)

    if not directory.exists():
        raise FileNotFoundError(
            f"FITS directory does not exist: {directory}"
        )

    if not directory.is_dir():
        raise NotADirectoryError(
            f"The specified path is not a directory: {directory}"
        )

    valid_suffixes = {
        ".fits",
        ".fts",
    }

    fits_files = [
        path
        for path in directory.iterdir()
        if (
            path.is_file()
            and not path.name.startswith(".")
            and path.suffix.lower() in valid_suffixes
        )
    ]

    fits_files.sort(
        key=lambda path: path.name.casefold()
    )

    if len(fits_files) == 0:
        raise FileNotFoundError(
            f"No FITS files were found in: {directory}"
        )

    return fits_files


# ============================================================
# Get observation times from the FITS header
# ============================================================

def get_observation_time(header):
    """
    Read the exposure start time, midpoint time, and exposure
    duration from a FITS header.

    DATE-OBS is used preferentially. DATEOBS is used as a
    fallback.

    Parameters
    ----------
    header : astropy.io.fits.Header
        FITS header.

    Returns
    -------
    time_start : astropy.time.Time
        Exposure start time in UTC.

    time_mid : astropy.time.Time
        Exposure midpoint time in UTC.

    exptime : float
        Exposure duration in seconds.
    """

    if "DATE-OBS" in header:
        date_obs = header["DATE-OBS"]
    elif "DATEOBS" in header:
        date_obs = header["DATEOBS"]
    else:
        raise KeyError(
            "The FITS header does not contain DATE-OBS "
            "or DATEOBS."
        )

    if "EXPTIME" not in header:
        raise KeyError(
            "The FITS header does not contain EXPTIME."
        )

    date_obs = str(date_obs).strip()

    if date_obs == "":
        raise ValueError(
            "DATE-OBS or DATEOBS is empty."
        )

    exptime = float(
        header["EXPTIME"]
    )

    if (
        not np.isfinite(exptime)
        or exptime <= 0
    ):
        raise ValueError(
            f"Invalid EXPTIME value: {exptime}"
        )

    # FITS timestamps are normally interpreted as UTC.
    try:
        time_start = Time(
            date_obs,
            format="fits",
            scale="utc",
        )
    except ValueError:
        # Use Astropy's automatic format detection as a fallback.
        time_start = Time(
            date_obs,
            scale="utc",
        )

    # Use the exposure midpoint as the representative timestamp.
    time_mid = (
        time_start
        + TimeDelta(
            exptime / 2.0,
            format="sec",
        )
    )

    return (
        time_start,
        time_mid,
        exptime,
    )


# ============================================================
# Refine the centroid around the previous position
# ============================================================

def refine_centroid_around_previous_position(
    image,
    previous_x,
    previous_y,
    half_size=20,
    sigma=3.0,
    maxiters=5,
):
    """
    Refine the source centroid around the previous-frame
    position using a two-dimensional Gaussian centroid.

    Parameters
    ----------
    image : numpy.ndarray
        Two-dimensional image array.

    previous_x, previous_y : float
        Source position measured in the previous frame.

    half_size : int, optional
        Half-size of the cutout region in pixels.

    sigma : float, optional
        Sigma-clipping threshold for background estimation.

    maxiters : int, optional
        Maximum number of sigma-clipping iterations.

    Returns
    -------
    x_centroid, y_centroid : float
        Refined centroid coordinates in the original image.
    """

    # Convert the image to a floating-point array.
    image = np.asarray(image)

    try:
        image = image.astype(
            np.float64,
            copy=False,
        )
    except (TypeError, ValueError) as error:
        raise TypeError(
            "The image data could not be converted to float. "
            f"dtype={image.dtype}"
        ) from error

    if image.ndim != 2:
        raise ValueError(
            "The image is not two-dimensional: "
            f"shape={image.shape}"
        )

    # Convert the previous centroid coordinates to scalar floats.
    try:
        previous_x = float(
            np.asarray(previous_x).squeeze()
        )
        previous_y = float(
            np.asarray(previous_y).squeeze()
        )
    except (TypeError, ValueError) as error:
        raise TypeError(
            "The previous-frame centroid coordinates could "
            "not be converted to float. "
            f"previous_x={previous_x!r}, "
            f"previous_y={previous_y!r}"
        ) from error

    if not (
        np.isfinite(previous_x)
        and np.isfinite(previous_y)
    ):
        raise ValueError(
            "The previous-frame centroid coordinates are not "
            "finite. "
            f"previous_x={previous_x}, "
            f"previous_y={previous_y}"
        )

    ny, nx = image.shape

    # Extract a cutout around the previous-frame position.
    x_center = int(
        round(previous_x)
    )
    y_center = int(
        round(previous_y)
    )

    x_min = max(
        0,
        x_center - half_size,
    )
    x_max = min(
        nx,
        x_center + half_size + 1,
    )

    y_min = max(
        0,
        y_center - half_size,
    )
    y_max = min(
        ny,
        y_center + half_size + 1,
    )

    cutout = image[
        y_min:y_max,
        x_min:x_max,
    ].copy()

    if cutout.size == 0:
        raise ValueError(
            "The centroid cutout is empty."
        )

    # Check the number of finite pixels.
    finite_mask = np.isfinite(
        cutout
    )

    if np.count_nonzero(finite_mask) < 9:
        raise ValueError(
            "The centroid cutout contains too few finite pixels."
        )

    # Estimate the local background.
    sigma_clip = SigmaClip(
        sigma=sigma,
        maxiters=maxiters,
    )

    clipped = sigma_clip(
        cutout[finite_mask],
        masked=True,
    )

    background = np.ma.median(
        clipped
    )

    if np.ma.is_masked(background):
        raise ValueError(
            "Background estimation failed because all pixels "
            "in the cutout were masked."
        )

    try:
        background = float(
            background
        )
    except (TypeError, ValueError) as error:
        raise TypeError(
            "The estimated background could not be converted "
            f"to float: {background!r}"
        ) from error

    if not np.isfinite(background):
        raise ValueError(
            "The estimated background is not finite: "
            f"{background}"
        )

    # Subtract the estimated background.
    source_cutout = (
        cutout.astype(
            np.float64,
            copy=True,
        )
        - background
    )

    source_cutout[
        ~np.isfinite(source_cutout)
    ] = 0.0

    # Negative values are not passed to centroid_2dg.
    source_cutout[
        source_cutout < 0
    ] = 0.0

    total_signal = float(
        np.sum(source_cutout)
    )

    if (
        not np.isfinite(total_signal)
        or total_signal <= 0
    ):
        raise ValueError(
            "No valid positive signal remains after background "
            "subtraction."
        )

    # Calculate the local source centroid.
    with warnings.catch_warnings():
        warnings.simplefilter(
            "ignore"
        )

        centroid_result = centroid_2dg(
            source_cutout
        )

    if centroid_result is None:
        raise ValueError(
            "centroid_2dg did not return a centroid."
        )

    try:
        x_local = float(
            np.asarray(
                centroid_result[0]
            ).squeeze()
        )

        y_local = float(
            np.asarray(
                centroid_result[1]
            ).squeeze()
        )

    except (
        TypeError,
        ValueError,
        IndexError,
    ) as error:
        raise TypeError(
            "The local centroid result could not be converted "
            f"to float: {centroid_result!r}"
        ) from error

    if not (
        np.isfinite(x_local)
        and np.isfinite(y_local)
    ):
        raise ValueError(
            "The local centroid coordinates are not finite. "
            f"x_local={x_local}, "
            f"y_local={y_local}"
        )

    # Convert the local coordinates to full-image coordinates.
    x_centroid = float(
        x_min + x_local
    )
    y_centroid = float(
        y_min + y_local
    )

    if not (
        0 <= x_centroid < nx
        and 0 <= y_centroid < ny
    ):
        raise ValueError(
            "The calculated centroid is outside the image. "
            f"x={x_centroid}, "
            f"y={y_centroid}, "
            f"image_shape={image.shape}"
        )

    return (
        x_centroid,
        y_centroid,
    )


# ============================================================
# Perform aperture photometry
# ============================================================

def measure_object(
    image,
    x_object,
    y_object,
    aperture_radius,
    annulus_r_in,
    annulus_r_out,
):
    """
    Perform aperture photometry for an artificial object.

    Parameters
    ----------
    image : numpy.ndarray
        Two-dimensional image array.

    x_object, y_object : float
        Object centroid coordinates.

    aperture_radius : float
        Aperture radius in pixels.

    annulus_r_in, annulus_r_out : float
        Inner and outer radii of the background annulus in
        pixels.

    Returns
    -------
    result : dict
        Aperture sum, background estimate, aperture area, and
        background-subtracted object flux.
    """

    image = np.asarray(
        image,
        dtype=float,
    )

    position = (
        float(x_object),
        float(y_object),
    )

    aperture = CircularAperture(
        position,
        r=aperture_radius,
    )

    annulus = CircularAnnulus(
        position,
        r_in=annulus_r_in,
        r_out=annulus_r_out,
    )

    # Measure the total signal inside the source aperture.
    phot_table = aperture_photometry(
        image,
        aperture,
        method="exact",
    )

    aperture_sum = float(
        phot_table[
            "aperture_sum"
        ][0]
    )

    # Estimate the background level in the annulus.
    sigma_clip = SigmaClip(
        sigma=3.0,
        maxiters=5,
    )

    annulus_stats = ApertureStats(
        image,
        annulus,
        sigma_clip=sigma_clip,
    )

    background_median = float(
        annulus_stats.median
    )

    if not np.isfinite(
        background_median
    ):
        raise ValueError(
            "The estimated annulus background is not finite."
        )

    # Calculate the geometrical area of the circular aperture.
    aperture_area = float(
        aperture.area
    )

    background_sum = (
        background_median
        * aperture_area
    )

    # Calculate the background-subtracted source flux in ADU.
    object_flux = (
        aperture_sum
        - background_sum
    )

    return {
        "aperture_sum": aperture_sum,
        "background_median": background_median,
        "background_sum": background_sum,
        "aperture_area": aperture_area,
        "object_flux": object_flux,
    }

#### JSON output

In [ ]:
def _to_json_value(value):
    """
    NumPy型、Astropy Tableの値、masked値などを
    JSONへ保存可能なPython標準型へ変換する。
    """

    # Astropy / NumPy masked value
    if np.ma.is_masked(value):
        return None

    # NumPy scalar
    if isinstance(value, np.generic):
        value = value.item()

    # bytes
    if isinstance(value, bytes):
        return value.decode("utf-8", errors="replace")

    # float
    if isinstance(value, float):
        if not np.isfinite(value):
            return None
        return float(value)

    # int
    if isinstance(value, (int, np.integer)):
        return int(value)

    # bool
    if isinstance(value, (bool, np.bool_)):
        return bool(value)

    if value is None:
        return None

    return value


def _get_column_value(row, column_name, default=None):
    """
    Astropy Tableの行から値を安全に取得する。
    """

    if column_name not in row.colnames:
        return default

    value = row[column_name]
    value = _to_json_value(value)

    if value is None:
        return default

    return value


def _normalise_iso_datetime(value):
    """
    ISO/FITS形式の時刻文字列をISO-8601形式へ整形する。

    例
    2026-07-06T12:34:56.123456
    """

    if value is None:
        return None

    value = str(value).strip()

    if value == "":
        return None

    try:
        time_value = Time(value, scale="utc")
        return time_value.isot
    except Exception:
        # Astropyで解釈できない場合は元の文字列を残す
        return value


def export_kupt_lightcurve_json(
    lightcurve_table,
    output_path,
    *,
    object_name,
    observer_name,
    observatory,
    aperture_radius_pixel,
    annulus_r_in_pixel,
    annulus_r_out_pixel,
    intl_des=None,
    norad_id=None,
    fits_header=None,
    magzero_data=None,
    editor_app_name="SatPhotometry bright LEO",
    editor_version="1.0.0",
    creator_app_name="SatPhotometry bright LEO",
    creator_version="1.0.0",
    creator_built=None,
    whole_comments=None,
    tle=None,
):
    """
    lightcurve_tableをKUPT-lightcurveJSON version 2形式で保存する。

    Parameters
    ----------
    lightcurve_table : astropy.table.Table
        測光結果のテーブル。

    output_path : str or pathlib.Path
        出力するJSONファイルのパス。

    object_name : str
        人工天体名。

    observer_name : str
        観測者名またはプロジェクト名。

    observatory : str
        観測所・観測装置名。

    aperture_radius_pixel : float
        天体測光開口半径 [pixel]。

    annulus_r_in_pixel, annulus_r_out_pixel : float
        背景測定用annulusの内半径・外半径 [pixel]。

    intl_des : str, optional
        国際標識。

    norad_id : str or int, optional
        NORAD Catalog ID。

    fits_header : astropy.io.fits.Header or dict, optional
        撮像設定を取得するための代表FITSヘッダー。
        通常は1フレーム目のヘッダーを渡す。

    magzero_data : dict, optional
        Magzero JSONをjson.load()した辞書。

    whole_comments : list[str], optional
        光度曲線全体についてのコメント。

    tle : list[str], optional
        TLEの1行目と2行目。

    Returns
    -------
    output_data : dict
        JSONへ保存した辞書。
    """

    if len(lightcurve_table) == 0:
        raise ValueError("lightcurve_tableが空です。")

    output_path = Path(output_path)
    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    if fits_header is None:
        fits_header = {}

    if magzero_data is None:
        magzero_data = {}

    if whole_comments is None:
        whole_comments = []

    if tle is None:
        tle = []

    if creator_built is None:
        creator_built = int(
            datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S")
        )

    # ========================================================
    # 有効な時刻を持つフレームを抽出
    # ========================================================

    valid_time_rows = []

    for row in lightcurve_table:
        time_start = _get_column_value(
            row,
            "time_start_utc",
            default=None,
        )

        if time_start not in (None, ""):
            valid_time_rows.append(row)

    if len(valid_time_rows) == 0:
        raise ValueError(
            "time_start_utcが有効なフレームがありません。"
        )

    first_row = valid_time_rows[0]
    last_row = valid_time_rows[-1]

    # ========================================================
    # 開始時刻・終了時刻
    # ========================================================

    start_datetime = _normalise_iso_datetime(
        _get_column_value(
            first_row,
            "time_start_utc",
        )
    )

    last_start_string = _get_column_value(
        last_row,
        "time_start_utc",
    )

    last_exptime = _get_column_value(
        last_row,
        "exptime_seconds",
        default=None,
    )

    if last_exptime is None:
        last_exptime = fits_header.get(
            "EXPTIME",
            0.0,
        )

    last_exptime = float(last_exptime)

    last_start_time = Time(
        last_start_string,
        scale="utc",
    )

    end_time = last_start_time + TimeDelta(
        last_exptime,
        format="sec",
    )

    end_datetime = end_time.isot

    # ========================================================
    # Capture settings
    # ========================================================

    first_exptime = _get_column_value(
        first_row,
        "exptime_seconds",
        default=fits_header.get("EXPTIME"),
    )

    first_egainsav = _get_column_value(
        first_row,
        "egainsav_electron_per_adu",
        default=fits_header.get("EGAINSAV"),
    )

    capture_settings = {
        "Xnaxis": _to_json_value(
            fits_header.get("NAXIS1")
        ),
        "Ynaxis": _to_json_value(
            fits_header.get("NAXIS2")
        ),
        "Xbinning": _to_json_value(
            fits_header.get(
                "XBINNING",
                fits_header.get("XBIN", 1),
            )
        ),
        "Ybinning": _to_json_value(
            fits_header.get(
                "YBINNING",
                fits_header.get("YBIN", 1),
            )
        ),
        "exposure": _to_json_value(
            first_exptime
        ),
        "gain": _to_json_value(
            fits_header.get("GAIN")
        ),
        "egain": _to_json_value(
            fits_header.get("EGAIN")
        ),
        "egainsav": _to_json_value(
            first_egainsav
        ),
        "offset": _to_json_value(
            fits_header.get(
                "OFFSET",
                fits_header.get("BLKLEVEL"),
            )
        ),
        "actualTemp": _to_json_value(
            fits_header.get(
                "CCD-TEMP",
                fits_header.get("CCDTEMP"),
            )
        ),
        "setTemp": _to_json_value(
            fits_header.get(
                "SET-TEMP",
                fits_header.get("SETTEMP"),
            )
        ),
        "camName": _to_json_value(
            fits_header.get(
                "INSTRUME",
                fits_header.get("CAMNAME"),
            )
        ),
        "camID": _to_json_value(
            fits_header.get(
                "CAMID",
                fits_header.get("CAMERAID"),
            )
        ),
        "swcreate": _to_json_value(
            fits_header.get(
                "SWCREATE",
                fits_header.get("CREATOR"),
            )
        ),
    }

    # 必須でない項目について、値がないキーを削除する
    capture_settings = {
        key: value
        for key, value in capture_settings.items()
        if value is not None
    }

    # 仕様上必須の画像サイズは存在確認
    if "Xnaxis" not in capture_settings:
        raise KeyError(
            "captureSettings.Xnaxisに必要な"
            "FITSヘッダーのNAXIS1がありません。"
        )

    if "Ynaxis" not in capture_settings:
        raise KeyError(
            "captureSettings.Ynaxisに必要な"
            "FITSヘッダーのNAXIS2がありません。"
        )

    # ========================================================
    # Photometric parameters
    # ========================================================

    phot_params_source = magzero_data.get(
        "phot_params",
        magzero_data,
    )

    phot_params = {
        "pixel_area_arcsec2": _to_json_value(
            phot_params_source.get(
                "pixel_area_arcsec2"
            )
        ),
        "aperture_area_arcsec2": _to_json_value(
            phot_params_source.get(
                "aperture_area_arcsec2"
            )
        ),
        "ZP": _to_json_value(
            phot_params_source.get(
                "photometric_zeropoint",
                phot_params_source.get("ZP"),
            )
        ),
        "ZP_exp": _to_json_value(
            phot_params_source.get(
                "photometric_zeropoint_exp",
                phot_params_source.get("ZP_exp"),
            )
        ),
        "ZP_elec": _to_json_value(
            phot_params_source.get(
                "photometric_zeropoint_elec",
                phot_params_source.get("ZP_elec"),
            )
        ),
        "ZP_scatter": _to_json_value(
            phot_params_source.get(
                "zeropoint_scatter",
                phot_params_source.get("ZP_scatter"),
            )
        ),
    }

    phot_params = {
        key: value
        for key, value in phot_params.items()
        if value is not None
    }

    # ========================================================
    # 各フレームのlight curve data
    # ========================================================

    data_list = []

    for row_number, row in enumerate(lightcurve_table):
        frame_id = _get_column_value(
            row,
            "frame_index",
            default=row_number,
        )

        status = _get_column_value(
            row,
            "status",
            default="Unknown",
        )

        # 以前のコードではstatusがsuccessだったため、
        # 仕様書の例に合わせてSuccessへ整形
        if isinstance(status, str):
            if status.lower() == "success":
                status = "Success"
            elif status.lower() == "non_positive_flux":
                status = "Non-positive flux"

        frame_comments = []

        filename = _get_column_value(
            row,
            "filename",
            default=None,
        )

        detection_method = _get_column_value(
            row,
            "detection_method",
            default=None,
        )

        if filename is not None:
            frame_comments.append(
                f"Source FITS file: {filename}"
            )

        if detection_method is not None:
            frame_comments.append(
                f"Detection method: {detection_method}"
            )

        light_curve_data = {
            "id": int(frame_id),
            "time_start_utc": _normalise_iso_datetime(
                _get_column_value(
                    row,
                    "time_start_utc",
                )
            ),
            "time_mid_utc": _normalise_iso_datetime(
                _get_column_value(
                    row,
                    "time_mid_utc",
                )
            ),
            "elapsed_seconds": _get_column_value(
                row,
                "elapsed_seconds",
            ),
            "x_centroid": _get_column_value(
                row,
                "x_centroid",
            ),
            "y_centroid": _get_column_value(
                row,
                "y_centroid",
            ),
            "aperture_sum": _get_column_value(
                row,
                "aperture_sum_adu",
            ),
            "bkg_med_pix": _get_column_value(
                row,
                "background_median_adu_per_pixel",
            ),
            "object_flux": _get_column_value(
                row,
                "object_flux_adu",
            ),
            "mag_in": _get_column_value(
                row,
                "instrumental_mag",
            ),
            "mag_app": _get_column_value(
                row,
                "apparent_mag",
            ),
            "status": status,
            "comment": {
                "general": frame_comments
            },
        }

        data_list.append(
            light_curve_data
        )

    # ========================================================
    # Top-level JSON
    # ========================================================

    output_data = {
        "header": {
            "object": {
                "objectName": str(object_name),
                "intlDES": (
                    None
                    if intl_des is None
                    else str(intl_des)
                ),
                "noradID": (
                    None
                    if norad_id is None
                    else str(norad_id)
                ),
            },
            "startDateTime": start_datetime,
            "endDateTime": end_datetime,
            "length": int(len(lightcurve_table)),
            "captureSettings": capture_settings,
            "photParams": phot_params,
            "observer": {
                "observerName": str(observer_name),
                "observatory": str(observatory),
            },
            "editor": {
                "appName": str(editor_app_name),
                "version": str(editor_version),
            },
            "measureSettings": {
                "measureFieldShape": "CIRCLE",
                "measureFieldSize": {
                    "aperture_radius_pixel": float(
                        aperture_radius_pixel
                    ),
                    "annulus_r_in_pixel": float(
                        annulus_r_in_pixel
                    ),
                    "annulus_r_out_pixel": float(
                        annulus_r_out_pixel
                    ),
                },
            },
        },
        "data": data_list,
        "createdBy": {
            "appName": str(creator_app_name),
            "version": str(creator_version),
            "built": int(creator_built),
        },
        "dataType": "KUPT-lightcurveJSON",
        "version": 2,
        "comment": {
            "general": [
                str(comment)
                for comment in whole_comments
            ],
            "TLE": [
                str(line)
                for line in tle
            ],
        }
    }

    # ========================================================
    # JSON保存
    # ========================================================

    with output_path.open(
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            output_data,
            file,
            ensure_ascii=False,
            indent=2,
            allow_nan=False,
        )

    print(
        f"KUPT-lightcurveJSONを保存しました: "
        f"{output_path}"
    )

    return output_data

#### Light curve

In [ ]:
# ============================================================
# Create the light curve
# ============================================================

def create_lightcurve(
    fits_directory,
    magzero_path,
    output_path,

    # Artificial-object information
    object_name,
    intl_des,
    norad_id,

    # Observer and observatory information
    observer_name,
    observatory,

    # Photometric aperture settings
    aperture_radius,
    annulus_r_in,
    annulus_r_out,

    # Processing software information
    editor_app_name="SatPhotometry bright LEO",
    editor_version="1.0.0",

    # JSON creation software information
    creator_app_name="SatPhotometry bright LEO",
    creator_version="1.0.0",

    # Optional metadata
    whole_comments=None,
    tle=None,

    # Source redetection setting
    redetect_on_failure=True,
):
    """
    Perform aperture photometry on all FITS files in a
    directory and save the resulting light curve as both CSV
    and KUPT-lightcurve JSON files.

    Parameters
    ----------
    fits_directory : str or pathlib.Path
        Directory containing the FITS images.

    magzero_path : str or pathlib.Path
        Path to the photometric zeropoint JSON file.

    output_path : str or pathlib.Path
        Base output path without a file extension.

    Returns
    -------
    result_table : astropy.table.Table
        Light-curve measurement table.

    output_path : str
        Final output path without a file extension.

    kupt_json_data
        Data returned by export_kupt_lightcurve_json().
    """

    # --------------------------------------------------------
    # Get the list of FITS files
    # --------------------------------------------------------

    fits_files = get_fits_files(
        fits_directory
    )

    print(
        f"Processing {len(fits_files)} FITS files."
    )

    # --------------------------------------------------------
    # Read the header of the first FITS file
    # --------------------------------------------------------

    with fits.open(
        fits_files[0],
        memmap=False,
    ) as hdul:
        first_fits_header = (
            hdul[0].header.copy()
        )

    # --------------------------------------------------------
    # Read the photometric zeropoint JSON file
    # --------------------------------------------------------

    with open(
        magzero_path,
        "r",
        encoding="utf-8",
    ) as file:
        magzero_data = json.load(
            file
        )

    photometric_zeropoint_elec = float(
        magzero_data[
            "phot_params"
        ][
            "photometric_zeropoint_elec"
        ]
    )

    if not np.isfinite(
        photometric_zeropoint_elec
    ):
        raise ValueError(
            "photometric_zeropoint_elec is not finite: "
            f"{photometric_zeropoint_elec}"
        )

    # --------------------------------------------------------
    # Initialize the result list and tracking position
    # --------------------------------------------------------

    results = []

    previous_x = None
    previous_y = None

    first_time_mid = None

    # --------------------------------------------------------
    # Process each FITS frame
    # --------------------------------------------------------

    for frame_index, fits_path in enumerate(
        fits_files
    ):
        print(
            f"[{frame_index + 1:04d}/"
            f"{len(fits_files):04d}] "
            f"{fits_path.name}"
        )

        status = "success"
        detection_method = ""

        try:
            # ------------------------------------------------
            # Read the FITS image and header
            # ------------------------------------------------

            with fits.open(
                fits_path,
                memmap=False,
            ) as hdul:
                img_header = (
                    hdul[0].header.copy()
                )

                img_data = np.asarray(
                    hdul[0].data,
                    dtype=float,
                )

            if img_data.ndim != 2:
                raise ValueError(
                    "The FITS image is not two-dimensional: "
                    f"shape={img_data.shape}"
                )

            # ------------------------------------------------
            # Read the exposure times
            # ------------------------------------------------

            (
                time_start,
                time_mid,
                exptime,
            ) = get_observation_time(
                img_header
            )

            if first_time_mid is None:
                first_time_mid = time_mid

            elapsed_seconds = float(
                (
                    time_mid
                    - first_time_mid
                ).to_value("sec")
            )

            # ------------------------------------------------
            # Determine whether a previous position is valid
            # ------------------------------------------------

            has_previous_position = False

            if (
                previous_x is not None
                and previous_y is not None
            ):
                try:
                    previous_x_float = float(
                        previous_x
                    )
                    previous_y_float = float(
                        previous_y
                    )

                    has_previous_position = (
                        np.isfinite(
                            previous_x_float
                        )
                        and np.isfinite(
                            previous_y_float
                        )
                    )

                except (
                    TypeError,
                    ValueError,
                ):
                    has_previous_position = False

            # ------------------------------------------------
            # Detect the artificial object
            # ------------------------------------------------

            if not has_previous_position:
                # Perform full-frame detection for the first
                # frame or after tracking has been interrupted.
                detection_result = (
                    detect_brightest_point_source(
                        img_data,
                        detection_sigma=8.0,
                        edge_sigma=5.0,
                        cutout_half_size=30,
                    )
                )

                x_object = float(
                    np.asarray(
                        detection_result[
                            "x_centroid"
                        ]
                    ).squeeze()
                )

                y_object = float(
                    np.asarray(
                        detection_result[
                            "y_centroid"
                        ]
                    ).squeeze()
                )

                detection_method = (
                    "full_frame_detection"
                )

            else:
                try:
                    # Search for the source in a local region
                    # around the previous-frame position.
                    detection_result = (
                        detect_source_around_previous_position(
                            img_data,
                            previous_x=previous_x,
                            previous_y=previous_y,
                            search_half_size=50,
                            detection_sigma=8.0,
                            edge_sigma=5.0,
                            detection_cutout_half_size=30,
                        )
                    )

                    x_object = float(
                        np.asarray(
                            detection_result[
                                "x_centroid"
                            ]
                        ).squeeze()
                    )

                    y_object = float(
                        np.asarray(
                            detection_result[
                                "y_centroid"
                            ]
                        ).squeeze()
                    )

                    detection_method = (
                        "local_source_detection"
                    )

                except Exception as local_error:
                    if not redetect_on_failure:
                        raise

                    print(
                        "    Local source detection failed. "
                        "Repeating full-frame detection."
                    )
                    print(
                        f"    Reason: {local_error}"
                    )

                    detection_result = (
                        detect_brightest_point_source(
                            img_data,
                            detection_sigma=8.0,
                            edge_sigma=5.0,
                            cutout_half_size=30,
                        )
                    )

                    x_object = float(
                        np.asarray(
                            detection_result[
                                "x_centroid"
                            ]
                        ).squeeze()
                    )

                    y_object = float(
                        np.asarray(
                            detection_result[
                                "y_centroid"
                            ]
                        ).squeeze()
                    )

                    detection_method = (
                        "full_frame_redetection"
                    )

            # ------------------------------------------------
            # Save the position for the next frame
            # ------------------------------------------------

            if (
                np.isfinite(x_object)
                and np.isfinite(y_object)
            ):
                previous_x = x_object
                previous_y = y_object

            else:
                previous_x = None
                previous_y = None

                raise ValueError(
                    "The detected artificial-object position "
                    "is not finite."
                )

            # ------------------------------------------------
            # Perform aperture photometry
            # ------------------------------------------------

            photometry = measure_object(
                image=img_data,
                x_object=x_object,
                y_object=y_object,
                aperture_radius=aperture_radius,
                annulus_r_in=annulus_r_in,
                annulus_r_out=annulus_r_out,
            )

            aperture_sum = photometry[
                "aperture_sum"
            ]

            background_median = photometry[
                "background_median"
            ]

            background_sum = photometry[
                "background_sum"
            ]

            aperture_area = photometry[
                "aperture_area"
            ]

            object_flux = photometry[
                "object_flux"
            ]

            # ------------------------------------------------
            # Read the detector gain
            # ------------------------------------------------

            if "EGAINSAV" not in img_header:
                raise KeyError(
                    "The FITS header does not contain EGAINSAV."
                )

            egainsav = float(
                img_header["EGAINSAV"]
            )

            if (
                not np.isfinite(egainsav)
                or egainsav <= 0
            ):
                raise ValueError(
                    f"Invalid EGAINSAV value: {egainsav}"
                )

            # ------------------------------------------------
            # Calculate the instrumental and apparent magnitude
            # ------------------------------------------------

            if (
                np.isfinite(object_flux)
                and object_flux > 0
            ):
                instrumental_mag = (
                    -2.5
                    * np.log10(
                        object_flux
                    )
                )

                apparent_mag = (
                    instrumental_mag
                    + 2.5 * np.log10(
                        exptime
                    )
                    - 2.5 * np.log10(
                        egainsav
                    )
                    + photometric_zeropoint_elec
                )

            else:
                instrumental_mag = np.nan
                apparent_mag = np.nan
                status = "non_positive_flux"

            # ------------------------------------------------
            # Store the frame result
            # ------------------------------------------------

            results.append(
                {
                    "frame_index": frame_index,
                    "filename": fits_path.name,
                    "time_start_utc": (
                        time_start.isot
                    ),
                    "time_mid_utc": (
                        time_mid.isot
                    ),
                    "elapsed_seconds": (
                        elapsed_seconds
                    ),
                    "exptime_seconds": (
                        exptime
                    ),
                    "egainsav_electron_per_adu": (
                        egainsav
                    ),
                    "x_centroid": (
                        x_object
                    ),
                    "y_centroid": (
                        y_object
                    ),
                    "detection_method": (
                        detection_method
                    ),
                    "aperture_radius_pixel": (
                        aperture_radius
                    ),
                    "annulus_r_in_pixel": (
                        annulus_r_in
                    ),
                    "annulus_r_out_pixel": (
                        annulus_r_out
                    ),
                    "aperture_area_pixel2": (
                        aperture_area
                    ),
                    "aperture_sum_adu": (
                        aperture_sum
                    ),
                    "background_median_adu_per_pixel": (
                        background_median
                    ),
                    "background_sum_adu": (
                        background_sum
                    ),
                    "object_flux_adu": (
                        object_flux
                    ),
                    "instrumental_mag": (
                        instrumental_mag
                    ),
                    "apparent_mag": (
                        apparent_mag
                    ),
                    "status": status,
                }
            )

            print(
                "    position = "
                f"({x_object:.3f}, {y_object:.3f})"
            )

            print(
                "    flux     = "
                f"{object_flux:.3f} ADU"
            )

            if np.isfinite(
                apparent_mag
            ):
                print(
                    "    mag      = "
                    f"{apparent_mag:.4f}"
                )
            else:
                print(
                    "    mag      = nan"
                )

        except Exception as error:
            print(
                f"    ERROR: {error}"
            )

            # Store an error row while preserving the table
            # column structure.
            results.append(
                {
                    "frame_index": frame_index,
                    "filename": fits_path.name,
                    "time_start_utc": "",
                    "time_mid_utc": "",
                    "elapsed_seconds": np.nan,
                    "exptime_seconds": np.nan,
                    "egainsav_electron_per_adu": np.nan,
                    "x_centroid": np.nan,
                    "y_centroid": np.nan,
                    "detection_method": (
                        detection_method
                    ),
                    "aperture_radius_pixel": (
                        aperture_radius
                    ),
                    "annulus_r_in_pixel": (
                        annulus_r_in
                    ),
                    "annulus_r_out_pixel": (
                        annulus_r_out
                    ),
                    "aperture_area_pixel2": np.nan,
                    "aperture_sum_adu": np.nan,
                    "background_median_adu_per_pixel": (
                        np.nan
                    ),
                    "background_sum_adu": np.nan,
                    "object_flux_adu": np.nan,
                    "instrumental_mag": np.nan,
                    "apparent_mag": np.nan,
                    "status": (
                        f"error: {error}"
                    ),
                }
            )

    # ========================================================
    # Create the output table
    # ========================================================

    result_table = Table(
        rows=results
    )

    # --------------------------------------------------------
    # Extract rows containing valid timestamps
    # --------------------------------------------------------

    valid_time_rows = []

    for row in results:
        time_string = row.get(
            "time_start_utc",
            "",
        )

        exptime_value = row.get(
            "exptime_seconds",
            np.nan,
        )

        if time_string is None:
            continue

        time_string = str(
            time_string
        ).strip()

        if time_string == "":
            continue

        try:
            exptime_value = float(
                exptime_value
            )
        except (
            TypeError,
            ValueError,
        ):
            continue

        if (
            not np.isfinite(exptime_value)
            or exptime_value <= 0
        ):
            continue

        try:
            frame_start_time = Time(
                time_string,
                format="isot",
                scale="utc",
            )
        except (
            TypeError,
            ValueError,
        ):
            try:
                frame_start_time = Time(
                    time_string,
                    scale="utc",
                )
            except (
                TypeError,
                ValueError,
            ):
                continue

        frame_end_time = (
            frame_start_time
            + TimeDelta(
                exptime_value,
                format="sec",
            )
        )

        valid_time_rows.append(
            {
                "start_time": (
                    frame_start_time
                ),
                "end_time": (
                    frame_end_time
                ),
                "filename": row.get(
                    "filename",
                    "",
                ),
            }
        )

    if len(valid_time_rows) == 0:
        raise RuntimeError(
            "No successfully processed frame contains a valid "
            "observation time. Check DATE-OBS, DATEOBS, "
            "EXPTIME, and the processing errors shown above."
        )

    # --------------------------------------------------------
    # Determine the observation start and end times
    # --------------------------------------------------------

    start_time = min(
        item["start_time"]
        for item in valid_time_rows
    )

    # Use the exposure end time of the final valid frame.
    end_time = max(
        item["end_time"]
        for item in valid_time_rows
    )

    start_datetime = (
        start_time.to_datetime()
    )

    end_datetime = (
        end_time.to_datetime()
    )

    # Format the observation date as YYYY-MM-DD.
    observation_date = (
        start_datetime.strftime(
            "%Y-%m-%d"
        )
    )

    # Format the start and end times as hhmmss.
    start_time_string = (
        start_datetime.strftime(
            "%H%M%S"
        )
    )

    end_time_string = (
        end_datetime.strftime(
            "%H%M%S"
        )
    )

    # Remove trailing underscores to prevent duplicated
    # separators in the final filename.
    output_path_base = str(
        output_path
    ).rstrip("_")

    output_path = (
        f"{output_path_base}_"
        f"{observation_date}_"
        f"{start_time_string}_"
        f"{end_time_string}"
    )

    output_csv_path = Path(
        output_path + ".csv"
    )

    output_json_path = Path(
        output_path + ".json"
    )

    output_csv_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    # --------------------------------------------------------
    # Save the light-curve table as CSV
    # --------------------------------------------------------

    result_table.write(
        output_csv_path,
        format="ascii.csv",
        overwrite=True,
    )

    print()
    print(
        "Saved the light-curve CSV file: "
        f"{output_csv_path}"
    )

    # --------------------------------------------------------
    # Save the light curve in KUPT-lightcurve JSON format
    # --------------------------------------------------------

    kupt_json_data = (
        export_kupt_lightcurve_json(
            lightcurve_table=result_table,
            output_path=output_json_path,

            # Artificial-object information
            object_name=object_name,
            intl_des=intl_des,
            norad_id=norad_id,

            # Observer and observatory information
            observer_name=observer_name,
            observatory=observatory,

            # Photometric aperture settings
            aperture_radius_pixel=(
                aperture_radius
            ),
            annulus_r_in_pixel=(
                annulus_r_in
            ),
            annulus_r_out_pixel=(
                annulus_r_out
            ),

            # FITS header and photometric zeropoint data
            fits_header=first_fits_header,
            magzero_data=magzero_data,

            # Processing software information
            editor_app_name=(
                editor_app_name
            ),
            editor_version=(
                editor_version
            ),

            # JSON creation software information
            creator_app_name=(
                creator_app_name
            ),
            creator_version=(
                creator_version
            ),

            # Optional metadata
            whole_comments=(
                whole_comments
            ),
            tle=tle,
        )
    )

    print(
        "Saved the KUPT-lightcurve JSON file: "
        f"{output_json_path}"
    )

    return (
        result_table,
        output_path,
        kupt_json_data,
    )

#### Run

In [ ]:
PATH_output_fname = (
    PATH_output
    + "lightcurve/"
    + f"lightcurve_{object_name}_{intl_des}"
)

(
    lightcurve_table,
    output_path,
    kupt_json_data,
) = create_lightcurve(
    fits_directory=PATH_fits,
    magzero_path=PATH_magzero,
    output_path=PATH_output_fname,

    # Artificial-object information
    object_name=object_name,
    intl_des=intl_des,
    norad_id=norad_id,

    # Observer and observatory information
    observer_name=observer_name,
    observatory=observatory,

    # Photometric aperture settings
    aperture_radius=aperture_radius,
    annulus_r_in=annulus_r_in,
    annulus_r_out=annulus_r_out,

    # Processing software information
    editor_app_name=(
        "SatPhotometry bright LEO"
    ),
    editor_version="1.0.0",

    # JSON creation software information
    creator_app_name=(
        "SatPhotometry bright LEO"
    ),
    creator_version="1.0.0",

    # Optional metadata
    whole_comments=None,
    tle=None,

    # Repeat full-frame detection if local detection fails
    redetect_on_failure=True,
)

### Result

#### Lightcurve

In [ ]:
elapsed_seconds = np.asarray(
    lightcurve_table["elapsed_seconds"],
    dtype=float,
)

apparent_mag = np.asarray(
    lightcurve_table["apparent_mag"],
    dtype=float,
)

valid = (
    np.isfinite(elapsed_seconds)
    & np.isfinite(apparent_mag)
)

plt.figure(figsize=(9, 5))

plt.plot(
    elapsed_seconds[valid],
    apparent_mag[valid],
    marker="o",
    markersize=4,
    linewidth=1,
)

plt.xlabel("Elapsed time from first frame [s]")
plt.ylabel("Apparent magnitude [mag]")

plt.gca().invert_yaxis()

plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()